In [ ]:
!pip install -q transformers sentencepiece tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# =========================================================
# XLM-RoBERTa + Evidential Deep Learning + MC Dropout (COLAB)
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (COLAB)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Malayalam"
RESULT_DIR = "/content/results_xlmr_edl"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_mal_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(
    SarcasmDataset(train_df, tokenizer),
    batch_size=8,
    shuffle=True
)

dev_loader = DataLoader(
    SarcasmDataset(dev_df, tokenizer),
    batch_size=16
)

test_loader = DataLoader(
    SarcasmDataset(test_df, tokenizer),
    batch_size=16
)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY FROM ALPHA
# -------------------------
def accuracy_from_alpha(alpha, labels):
    probs = alpha / alpha.sum(dim=1, keepdim=True)
    preds = probs.argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialXLMR(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialXLMR(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )

        labels = batch["labels"].to(DEVICE)
        loss = evidential_loss(alpha, labels, NUM_CLASSES)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)
            loss = evidential_loss(alpha, labels, NUM_CLASSES)

            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...


Epoch 7 | Train Loss: 0.1488, Train Acc: 0.9132 | Val Loss: 0.2916, Val Acc: 0.8147


Epoch 8 [VAL]: 100%|██████████| 189/189 [00:21<00:00,  8.97it/s]



Epoch 8 | Train Loss: 0.1323, Train Acc: 0.9242 | Val Loss: 0.2537, Val Acc: 0.8501


Epoch 9 [VAL]: 100%|██████████| 189/189 [00:21<00:00,  8.94it/s]



Epoch 9 | Train Loss: 0.1591, Train Acc: 0.9053 | Val Loss: 0.2571, Val Acc: 0.8392
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.8598726114649682
F1: 0.5368421052631579

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.95      0.92      3083
           1       0.67      0.45      0.54       685

    accuracy                           0.86      3768
   macro avg       0.78      0.70      0.73      3768
weighted avg       0.85      0.86      0.85      3768

Average Predictive Uncertainty: 0.004068454


DISTILL BERT


In [ ]:
# =========================================================
# DistilBERT-multilingual + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Malayalam"
RESULT_DIR = "/content/drive/MyDrive/results/DISTIL_M_EDL"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_mal_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "distilbert-base-multilingual-cased"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=16, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=32)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=32)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialDistilBERT(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]  # [CLS]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialDistilBERT(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 20
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)
            loss = evidential_loss(alpha, labels, NUM_CLASSES)

            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()  # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...


Epoch 3 | Train Loss: 0.1999, Train Acc: 0.8788 | Val Loss: 0.2636, Val Acc: 0.8326


Epoch 4 [VAL]: 100%|██████████| 95/95 [00:10<00:00,  8.68it/s]



Epoch 4 | Train Loss: 0.1596, Train Acc: 0.9070 | Val Loss: 0.2579, Val Acc: 0.8428


Epoch 5 [VAL]: 100%|██████████| 95/95 [00:10<00:00,  8.69it/s]



Epoch 5 | Train Loss: 0.1293, Train Acc: 0.9273 | Val Loss: 0.2796, Val Acc: 0.8352
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.8542993630573248
F1: 0.35789473684210527

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.99      0.92      3083
           1       0.90      0.22      0.36       685

    accuracy                           0.85      3768
   macro avg       0.88      0.61      0.64      3768
weighted avg       0.86      0.85      0.82      3768

Average Predictive Uncertainty: 0.0004178737


mBART

In [ ]:
# =========================================================
# mBART (facebook/mbart-large-50) + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Malayalam"
RESULT_DIR = "/content/drive/MyDrive/results/MBART_EDL"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_mal_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "facebook/mbart-large-50"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=4, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=8)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=8)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialMBART(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # <s>
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialMBART(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


Epoch 1 [TRAIN]:  87%|████████▋ | 2626/3015 [25:04<03:42,  1.75it/s]


KeyboardInterrupt: 

DeBERTa

In [ ]:
# =========================================================
# mDeBERTa-v3 + Evidential Deep Learning + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Malayalam"
RESULT_DIR = "/content/drive/MyDrive/results/MDEBERTA_EDL"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_mal_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "microsoft/mdeberta-v3-base"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=8, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=16)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=16)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialDeBERTa(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # CLS
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialDeBERTa(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ---- TRAIN ----
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    # ---- VALIDATION ----
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    # ---- EARLY STOPPING ----
    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Epoch 1 [VAL]: 100%|██████████| 189/189 [00:29<00:00,  6.50it/s]



Epoch 1 | Train Loss: 0.2828, Train Acc: 0.8321 | Val Loss: 0.2618, Val Acc: 0.8408


Epoch 2 [VAL]: 100%|██████████| 189/189 [00:29<00:00,  6.49it/s]



Epoch 2 | Train Loss: 0.2338, Train Acc: 0.8609 | Val Loss: 0.2304, Val Acc: 0.8587


...


Epoch 6 | Train Loss: 0.1130, Train Acc: 0.9370 | Val Loss: 0.2660, Val Acc: 0.8411


Epoch 7 [VAL]: 100%|██████████| 189/189 [00:29<00:00,  6.44it/s]



Epoch 7 | Train Loss: 0.1041, Train Acc: 0.9436 | Val Loss: 0.2381, Val Acc: 0.8534


Epoch 8 [VAL]: 100%|██████████| 189/189 [00:29<00:00,  6.51it/s]



Epoch 8 | Train Loss: 0.0957, Train Acc: 0.9482 | Val Loss: 0.2439, Val Acc: 0.8581
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.8742038216560509
F1: 0.5485714285714286

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.98      0.93      3083
           1       0.79      0.42      0.55       685

    accuracy                           0.87      3768
   macro avg       0.84      0.70      0.74      3768
weighted avg       0.87      0.87      0.86      3768

Average Predictive Uncertainty: 0.002569712


BERT


In [ ]:
# =========================================================
# Multilingual BERT + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Malayalam"
RESULT_DIR = "/content/drive/MyDrive/results/MBERT_EDL"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_mal_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "bert-base-multilingual-cased"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=16, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=32)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=32)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialBERT(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # [CLS]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialBERT(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


Epoch 1 [VAL]: 100%|██████████| 95/95 [00:21<00:00,  4.36it/s]



Epoch 1 | Train Loss: 0.2911, Train Acc: 0.8291 | Val Loss: 0.2831, Val Acc: 0.8314


Epoch 2 [VAL]: 100%|██████████| 95/95 [00:21<00:00,  4.45it/s]



Epoch 2 | Train Loss: 0.2686, Train Acc: 0.8387 | Val Loss: 0.2741, Val Acc: 0.8227


...


Epoch 6 | Train Loss: 0.2180, Train Acc: 0.8714 | Val Loss: 0.2804, Val Acc: 0.8409


Epoch 7 [VAL]: 100%|██████████| 95/95 [00:20<00:00,  4.58it/s]



Epoch 7 | Train Loss: 0.3009, Train Acc: 0.8250 | Val Loss: 0.3256, Val Acc: 0.8054


Epoch 8 [VAL]: 100%|██████████| 95/95 [00:20<00:00,  4.59it/s]



Epoch 8 | Train Loss: 0.3171, Train Acc: 0.8127 | Val Loss: 0.3248, Val Acc: 0.8054
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.8532377919320594
F1: 0.45517241379310347

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.97      0.92      3083
           1       0.70      0.34      0.46       685

    accuracy                           0.85      3768
   macro avg       0.78      0.65      0.69      3768
weighted avg       0.84      0.85      0.83      3768

Average Predictive Uncertainty: 0.0018712839


MiniLM

In [ ]:
# =========================================================
# MiniLM Multilingual + Evidential DL + MC Dropout
# Google Drive + Colab READY
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# DEVICE
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# PATHS (GOOGLE DRIVE)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/SARCASM DATASET /Malayalam"
RESULT_DIR = "/content/drive/MyDrive/results/MINILM_EDL"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_mal_test.csv")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "microsoft/multilingual-MiniLM-L12-H384"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=32, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=64)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=64)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY
# -------------------------
def accuracy_from_alpha(alpha, labels):
    preds = (alpha / alpha.sum(dim=1, keepdim=True)).argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialMiniLM(nn.Module):
    def __init__(self, num_classes=2, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # [CLS]
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        return evidence + 1

model = EvidentialMiniLM(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 50
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE)
        )
        labels = batch["labels"].to(DEVICE)

        loss = evidential_loss(alpha, labels, NUM_CLASSES)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, labels))

    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)

            loss = evidential_loss(alpha, labels, NUM_CLASSES)
            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, labels))

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {np.mean(train_losses):.4f}, "
        f"Train Acc: {np.mean(train_accs):.4f} | "
        f"Val Loss: {np.mean(val_losses):.4f}, "
        f"Val Acc: {np.mean(val_accs):.4f}"
    )

    if np.mean(val_accs) > best_val_acc:
        best_val_acc = np.mean(val_accs)
        wait = 0
        torch.save(model.state_dict(), f"{RESULT_DIR}/best_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(f"{RESULT_DIR}/best_model.pt"))
model.train()   # dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...


Epoch 3 | Train Loss: 0.3059, Train Acc: 0.8332 | Val Loss: 0.3425, Val Acc: 0.8059


Epoch 4 [VAL]: 100%|██████████| 48/48 [00:06<00:00,  7.14it/s]



Epoch 4 | Train Loss: 0.3109, Train Acc: 0.8201 | Val Loss: 0.3000, Val Acc: 0.8251


Epoch 5 [VAL]: 100%|██████████| 48/48 [00:06<00:00,  6.98it/s]



Epoch 5 | Train Loss: 0.3038, Train Acc: 0.8282 | Val Loss: 0.3136, Val Acc: 0.8059
⛔ Early stopping

================ TEST RESULTS ================
Accuracy: 0.8450106157112527
F1: 0.2963855421686747

Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.99      0.91      3083
           1       0.85      0.18      0.30       685

    accuracy                           0.85      3768
   macro avg       0.85      0.59      0.60      3768
weighted avg       0.85      0.85      0.80      3768

Average Predictive Uncertainty: 0.0012737568


In [ ]:
# =========================================================
# XLM-RoBERTa + Evidential Deep Learning + MC Dropout
# TRAIN / VAL ACCURACY + LOSS (FINAL CLEAN VERSION)
# =========================================================

import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -------------------------
# PATHS
# -------------------------
DATA_DIR = r"D:\PROJECT -MTech\Datasets_SARCASM\MALAYALAM"
RESULT_DIR = r"D:\PROJECT -MTech\S4\results\Transformer\MAL\new_code\XLMR_EDL"
os.makedirs(RESULT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "sarcasm_mal_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "sarcasm_mal_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "sarcasm_mal_test.csv")

DEVICE = torch.device("cpu")

# -------------------------
# SEED
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()

# -------------------------
# LOAD DATA
# -------------------------
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "Text"
LABEL_COL = "labels"

LABEL_MAP = {"Non-sarcastic": 0, "Sarcastic": 1}

def encode_labels(df):
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
    if df[LABEL_COL].isnull().any():
        raise ValueError("❌ Unmapped labels detected")
    return df

train_df = encode_labels(train_df)
dev_df   = encode_labels(dev_df)
test_df  = encode_labels(test_df)

# -------------------------
# DATASET
# -------------------------
class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# -------------------------
# TOKENIZER & LOADERS
# -------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_CLASSES = 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_loader = DataLoader(SarcasmDataset(train_df, tokenizer), batch_size=8, shuffle=True)
dev_loader   = DataLoader(SarcasmDataset(dev_df, tokenizer), batch_size=16)
test_loader  = DataLoader(SarcasmDataset(test_df, tokenizer), batch_size=16)

# -------------------------
# EVIDENTIAL LOSS
# -------------------------
def evidential_loss(alpha, target, num_classes):
    S = torch.sum(alpha, dim=1, keepdim=True)
    probs = alpha / S
    one_hot = F.one_hot(target, num_classes=num_classes).float()
    mse = torch.sum((one_hot - probs) ** 2, dim=1)
    var = torch.sum(probs * (1 - probs) / (S + 1), dim=1)
    return torch.mean(mse + var)

# -------------------------
# ACCURACY FROM ALPHA
# -------------------------
def accuracy_from_alpha(alpha, labels):
    probs = alpha / alpha.sum(dim=1, keepdim=True)
    preds = probs.argmax(dim=1)
    return (preds == labels).float().mean().item()

# -------------------------
# MODEL
# -------------------------
class EvidentialXLMR(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]   # <s> token
        cls = self.dropout(cls)
        evidence = F.softplus(self.fc(cls))
        alpha = evidence + 1
        return alpha

model = EvidentialXLMR(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# TRAINING + VALIDATION
# -------------------------
EPOCHS = 30
PATIENCE = 3

best_val_acc = 0.0
wait = 0

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_losses, train_accs = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]"):
        optimizer.zero_grad()

        alpha = model(batch["input_ids"], batch["attention_mask"])
        loss = evidential_loss(alpha, batch["labels"], NUM_CLASSES)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_accs.append(accuracy_from_alpha(alpha, batch["labels"]))

    avg_train_loss = np.mean(train_losses)
    avg_train_acc  = np.mean(train_accs)

    # ===== VALIDATION =====
    model.eval()
    val_losses, val_accs = [], []

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc=f"Epoch {epoch+1} [VAL]"):
            alpha = model(batch["input_ids"], batch["attention_mask"])
            loss = evidential_loss(alpha, batch["labels"], NUM_CLASSES)

            val_losses.append(loss.item())
            val_accs.append(accuracy_from_alpha(alpha, batch["labels"]))

    avg_val_loss = np.mean(val_losses)
    avg_val_acc  = np.mean(val_accs)

    print(
        f"\nEpoch {epoch+1} | "
        f"Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}"
    )

    # ===== EARLY STOPPING =====
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        wait = 0
        torch.save(model.state_dict(), os.path.join(RESULT_DIR, "best_model.pt"))
    else:
        wait += 1
        if wait >= PATIENCE:
            print("⛔ Early stopping triggered")
            break

# -------------------------
# FINAL TEST (MC DROPOUT)
# -------------------------
model.load_state_dict(torch.load(os.path.join(RESULT_DIR, "best_model.pt")))
model.train()   # keep dropout ON

all_preds, all_uncert, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        mc_probs = []

        for _ in range(10):
            alpha = model(batch["input_ids"], batch["attention_mask"])
            probs = alpha / alpha.sum(dim=1, keepdim=True)
            mc_probs.append(probs.unsqueeze(0))

        mc_probs = torch.cat(mc_probs, dim=0)
        mean_probs = mc_probs.mean(dim=0)
        uncertainty = mc_probs.var(dim=0).mean(dim=1)

        all_preds.extend(mean_probs.argmax(dim=1).cpu().numpy())
        all_uncert.extend(uncertainty.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

print("\n================ TEST RESULTS ================")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))
print("Average Predictive Uncertainty:", np.mean(all_uncert))
